# TS21 Skeletonization

## Setup

In [3]:
from mcf2swc import *

from swctools import SWCModel, FrustaSet, PointSet, plot_model

import logging
logging.basicConfig(level=logging.INFO)

import os

print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


## Load Mesh and skeleton

In [4]:
spine_idx = 21
mcf_qst = 0.1
mcf_mcst = 1
obj_name = f"TS{spine_idx}"
skeleton_name = f"TS{spine_idx}_qst{mcf_qst}_mcst{mcf_mcst}"
mm = MeshManager(mesh_path=f"../data/mesh/processed/{obj_name}_simplified.obj")
raw_skeleton = SkeletonGraph.from_txt(f"../data/mcf_skeletons/{skeleton_name}.polylines.txt")
print(raw_skeleton)
# pls.prune_short_branches_inplace(min_length=10)
mm.visualize_mesh_3d(skel=raw_skeleton)

INFO:mcf2swc.mesh:Loaded mesh: 2017 vertices, 4040 faces


SkeletonGraph with 198 nodes and 198 edges


# Skeleton optimization

In [5]:
from mcf2swc import (
    SkeletonGraph,
    MeshManager,
    SkeletonOptimizer,
    SkeletonOptimizerOptions,
)

do_skeleton_optimization = True

if do_skeleton_optimization:
    opts = SkeletonOptimizerOptions(
        max_iterations=10,
        step_size=1.0,
        smoothing_weight=0.1,
        verbose=True,
    )
    # Create optimizer and check for surface crossing
    optimizer = SkeletonOptimizer(raw_skeleton, mm.mesh, opts)
    has_crossing, num_outside, max_dist = optimizer.check_surface_crossing()
    print(f"Surface crossing: {has_crossing}")
    print(f"Points outside: {num_outside}/{raw_skeleton.total_points()}")
    print(f"Max distance: {max_dist:.4f}")
    # Run optimization
    print("\nOptimizing skeleton...")
    optimized_skeleton = optimizer.optimize()

    skeleton = optimized_skeleton
    all_skeletons = [raw_skeleton, optimized_skeleton]
else:
    skeleton = raw_skeleton
    all_skeletons = [raw_skeleton]


mm.visualize_mesh_3d(skel=all_skeletons, skel_color=["blue", "crimson"])


INFO:mcf2swc.skeleton_optimizer:Surface crossing detected: 47/198 nodes outside mesh (max distance: 12.8815)


Surface crossing: True
Points outside: 47/198
Max distance: 12.8815

Optimizing skeleton...


INFO:mcf2swc.skeleton_optimizer:Surface crossing detected: 47/198 nodes outside mesh (max distance: 12.8815)
INFO:mcf2swc.skeleton_optimizer:Starting skeleton optimization...
INFO:mcf2swc.skeleton_optimizer:  Nodes: 198
INFO:mcf2swc.skeleton_optimizer:  Max iterations: 10
INFO:mcf2swc.skeleton_optimizer:  Step size: 1.0000
INFO:mcf2swc.skeleton_optimizer:  Smoothing weight: 0.1000
INFO:mcf2swc.skeleton_optimizer:  Iteration 0: avg movement = 0.905004
INFO:mcf2swc.skeleton_optimizer:Optimization complete


# Fit morphology

In [8]:
spacing = 50

swc_out_dir = f"../data/swc/current/{skeleton_name}"

# check if directory exists, if not create it
if not os.path.exists(swc_out_dir):
    os.makedirs(swc_out_dir)

radius_strategy_list = [
    "equivalent_area",
    # "equivalent_perimeter",
    # "section_median",
    # "section_circle_fit",
    # "nearest_surface",
]
for radius_strategy in radius_strategy_list:
    print(f"Computing skeleton for radius_strategy={radius_strategy} ...", end="")
    morph = fit_morphology(
        mm.mesh, 
        skeleton, 
        options=FitOptions(
        spacing=spacing,
        radius_strategy=radius_strategy,
        snap_polylines_to_mesh=False)
)
    # write swc to file
    morph.to_swc_file(f"{swc_out_dir}/TS{spine_idx}_s{spacing}_{radius_strategy}.swc")
    morph.print_attributes(node_info=False, edge_info=False)
    print("DONE")

INFO:mcf2swc.graph_fitting:Tracing start: mesh[V=2017,F=4040], skeleton[nodes=198,edges=198], spacing=50, radius_strategy=equivalent_area
INFO:mcf2swc.graph_fitting:Edge group 0: input_pts=32 -> samples=5
INFO:mcf2swc.graph_fitting:Edge group 1: input_pts=23 -> samples=4
INFO:mcf2swc.graph_fitting:Edge group 2: input_pts=10 -> samples=3


Computing skeleton for radius_strategy=equivalent_area ...

INFO:mcf2swc.graph_fitting:Edge group 3: input_pts=21 -> samples=4
INFO:mcf2swc.graph_fitting:Edge group 4: input_pts=5 -> samples=2
INFO:mcf2swc.graph_fitting:Edge group 5: input_pts=17 -> samples=4
INFO:mcf2swc.graph_fitting:Edge group 6: input_pts=3 -> samples=2
INFO:mcf2swc.graph_fitting:Edge group 7: input_pts=7 -> samples=2
INFO:mcf2swc.graph_fitting:Edge group 8: input_pts=7 -> samples=2
INFO:mcf2swc.graph_fitting:Edge group 9: input_pts=19 -> samples=4
INFO:mcf2swc.graph_fitting:Edge group 10: input_pts=22 -> samples=4
INFO:mcf2swc.graph_fitting:Edge group 11: input_pts=34 -> samples=6
INFO:mcf2swc.graph_fitting:Edge group 12: input_pts=2 -> samples=2
INFO:mcf2swc.graph_fitting:Edge group 13: input_pts=10 -> samples=3
INFO:mcf2swc.graph_fitting:Tracing done: nodes=33, edges=33, samples=47, section=47, fallback=0


MorphologyGraph: nodes=33, edges=33, components=1, cycles=1, branch_points=7, leaves=7, self_loops=0, density=0.0625
DONE


# Plotting

In [9]:
# plot using swctools
make_html = False

for radius_strategy in radius_strategy_list:
    swc_filepath = f"{swc_out_dir}/TS{spine_idx}_s{spacing}_{radius_strategy}.swc"
    model = SWCModel.from_swc_file(swc_filepath)
    model.print_attributes(node_info=False, edge_info=False)
    frusta = FrustaSet.from_swc_model(model)
    title = f"TS{spine_idx}_s{spacing}_{radius_strategy}"
    fig = plot_model(swc_model=model, frusta=frusta, slider=True, title=title)
    fig.show()
    if make_html:
        fig.write_html(f"../plotly/TS{spine_idx}_s{spacing}_{radius_strategy}.html")


INFO:swctools.io:parse_swc start strict=True validate_reconnections=True float_tol=1e-09
INFO:swctools.io:parse_swc done records=34 reconnections=1 header=5
INFO:swctools.model:SWCModel.from_parse_result records=34 reconnections=1 header=5
INFO:swctools.model:SWCModel.from_swc_file built nodes=34 edges=33 strict=True validate_reconnections=True
INFO:swctools.geometry:batch_frusta count=33 sides=16 end_caps=False verts=1056 faces=1056
INFO:swctools.geometry:FrustaSet.from_swc_model edges=33 sides=16 end_caps=False
INFO:swctools.geometry:batch_frusta count=33 sides=16 end_caps=False verts=1056 faces=1056
INFO:swctools.geometry:FrustaSet.scaled radius_scale=0.0
INFO:swctools.geometry:batch_frusta count=33 sides=16 end_caps=False verts=1056 faces=1056
INFO:swctools.geometry:FrustaSet.scaled radius_scale=0.05
INFO:swctools.geometry:batch_frusta count=33 sides=16 end_caps=False verts=1056 faces=1056
INFO:swctools.geometry:FrustaSet.scaled radius_scale=0.1
INFO:swctools.geometry:batch_frusta 

SWCModel: nodes=34, edges=33, components=1, cycles=0, branch_points=6, roots=1, leaves=8, self_loops=0, density=0.0588


INFO:swctools.geometry:FrustaSet.scaled radius_scale=0.45
INFO:swctools.geometry:batch_frusta count=33 sides=16 end_caps=False verts=1056 faces=1056
INFO:swctools.geometry:FrustaSet.scaled radius_scale=0.5
INFO:swctools.geometry:batch_frusta count=33 sides=16 end_caps=False verts=1056 faces=1056
INFO:swctools.geometry:FrustaSet.scaled radius_scale=0.55
INFO:swctools.geometry:batch_frusta count=33 sides=16 end_caps=False verts=1056 faces=1056
INFO:swctools.geometry:FrustaSet.scaled radius_scale=0.6
INFO:swctools.geometry:batch_frusta count=33 sides=16 end_caps=False verts=1056 faces=1056
INFO:swctools.geometry:FrustaSet.scaled radius_scale=0.65
INFO:swctools.geometry:batch_frusta count=33 sides=16 end_caps=False verts=1056 faces=1056
INFO:swctools.geometry:FrustaSet.scaled radius_scale=0.7
INFO:swctools.geometry:batch_frusta count=33 sides=16 end_caps=False verts=1056 faces=1056
INFO:swctools.geometry:FrustaSet.scaled radius_scale=0.75
INFO:swctools.geometry:batch_frusta count=33 sides=